                   step1- 5 PDF files
                         ↓
                    step2-PDF Reader
                         ↓
                    step3-Text extraction
                         ↓
                      step4-Chunks
                         ↓
                    step5-Embeddings
                         ↓
                 step6-384-number vectors
                         ↓
              
            step7-QUESTION - Similarity Search
            
                         ↓
                  step8-Relevant chunk
                         ↓
               step9-benefits_policy.pdf
                         ↓
             step10- "90 days of employment"

5 PDF files
     ↓
PDF Reader (pypdf)
     ↓
Text extraction
     ↓
Chunks
     ↓
Sentence Transformer
(all-MiniLM-L6-v2)
     ↓
384-dimensional embeddings
     ↓
Python variable: embeddings
     ↓
Cosine Similarity
(sklearn)
     ↓
Top 3 relevant chunks
     ↓
Gemini
     ↓
Final answer

step1- 5 PDF files

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
# to read pdf files
!pip install pypdf

Save all PDF text into one .txt file

     step2-PDF Reader

In [ ]:

from pypdf import PdfReader
import os

pdf_files = [
    "/content/benefits_policy.pdf",
    "/content/expense_policy.pdf",
    "/content/code_of_conduct.pdf",
    "/content/remote_work_policy.pdf",
    "/content/vacation_policy.pdf"
]

for pdf_path in pdf_files:

    if os.path.exists(pdf_path):

        reader = PdfReader(pdf_path)

        print("=" * 60)
        print("PDF:", os.path.basename(pdf_path))
        print("Number of pages:", len(reader.pages))

        # Read the text from every page
        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text()

            print(f"\n--- Page {page_number} ---")
            print(text)

    else:
        print(f"PDF not found: {pdf_path}")

 step3-Text extraction

In [ ]:
#Save all PDF text into one .txt file
from pypdf import PdfReader
import os

pdf_files = [
    "/content/benefits_policy.pdf",
    "/content/expense_policy.pdf",
    "/content/code_of_conduct.pdf",
    "/content/remote_work_policy.pdf",
    "/content/vacation_policy.pdf"
]

output_file = "/content/all_policies.txt"

with open(output_file, "w", encoding="utf-8") as f:

    for pdf_path in pdf_files:

        if os.path.exists(pdf_path):

            reader = PdfReader(pdf_path)

            f.write("=" * 60 + "\n")
            f.write(f"PDF: {os.path.basename(pdf_path)}\n")
            f.write(f"Number of pages: {len(reader.pages)}\n")
            f.write("=" * 60 + "\n\n")

            for page_number, page in enumerate(reader.pages, start=1):

                text = page.extract_text()

                f.write(f"--- Page {page_number} ---\n")
                f.write(text or "")
                f.write("\n\n")

        else:
            print(f"PDF not found: {pdf_path}")

print("Done!")
print("Saved to:", output_file)

In [ ]:
#Step 2 — Check the saved file
with open("/content/all_policies.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:5000])

In [ ]:
#— Download it to your computer
from google.colab import files

files.download("/content/all_policies.txt")

In [ ]:
#Create chunks from your PDFs
from pypdf import PdfReader
import os
import re

pdf_files = [
    "/content/benefits_policy.pdf",
    "/content/expense_policy.pdf",
    "/content/code_of_conduct.pdf",
    "/content/remote_work_policy.pdf",
    "/content/vacation_policy.pdf"
]

chunks = []

chunk_size = 500
chunk_overlap = 100

for pdf_path in pdf_files:

    if os.path.exists(pdf_path):

        reader = PdfReader(pdf_path)

        for page_number, page in enumerate(reader.pages, start=1):

            text = page.extract_text()

            if not text:
                continue

            # Clean extra spaces
            text = " ".join(text.split())

            # Split into sentences
            sentences = re.split(r'(?<=[.!?])\s+', text)

            current_chunk = ""

            for sentence in sentences:

                # If adding the sentence keeps us under the chunk size
                if len(current_chunk) + len(sentence) <= chunk_size:

                    current_chunk += sentence + " "

                else:

                    # Save the current chunk
                    if current_chunk.strip():

                        chunks.append({
                            "text": current_chunk.strip(),
                            "source": os.path.basename(pdf_path),
                            "page": page_number
                        })

                    # Start the next chunk
                    current_chunk = sentence + " "

            # Save the final chunk
            if current_chunk.strip():

                chunks.append({
                    "text": current_chunk.strip(),
                    "source": os.path.basename(pdf_path),
                    "page": page_number
                })

    else:
        print(f"PDF not found: {pdf_path}")

print("Total chunks:", len(chunks))

   step4-Chunks

In [ ]:
#Look at your chunks
for i, chunk in enumerate(chunks[:5]):

    print("=" * 60)
    print("Chunk:", i)
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print()
    print(chunk["text"])

In [ ]:
#Before we create embeddings Let's save your chunks so we don't have to recreate them every time.
import json

with open("/content/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print("Saved:", len(chunks), "chunks")

In [ ]:
#Then verify:
import json

with open("/content/chunks.json", "r", encoding="utf-8") as f:
    saved_chunks = json.load(f)

print("Loaded chunks:", len(saved_chunks))
print(saved_chunks[0])

step5-Embeddings

EMBEDDING
First, understand the idea
An embedding converts text into a list of numbers that represents its meaning.
For example:
"Employees receive dental benefits."
              ↓
        Embedding model
              ↓
[0.21, -0.13, 0.87, 0.42, ...]

And:

"Does the company cover dental services?"
              ↓
        Embedding model
              ↓
[0.19, -0.11, 0.84, 0.45, ...]

Because these two sentences have similar meanings, their numerical representations should also be relatively close.

That's what will eventually allow your RAG system to find the correct policy chunk.
install the embedding library → load the model → create an embedding for your first chunk → inspect the numbers → then embed all 11 chunks.

In [ ]:
#Install the embedding library
!pip install -q sentence-transformers

In [ ]:
#Import the model,We loaded a pretrained embedding model.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

In [ ]:
#Create an embedding for ONE chunk,Why 384 numbers? The model we're using represents each piece of text as a 384-dimensional vector.
first_chunk = chunks[0]["text"]

embedding = model.encode(first_chunk)

print("Number of values:", len(embedding))
print("First 10 values:")
print(embedding[:10])

In [ ]:
# Embed ALL 11 chunks,Chunk 0 → 384 numbers Chunk 1 → 384 numbers,Chunk 2 → 384 numbers ...Chunk 10 → 384 numbers
texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(texts)

print("Number of chunks:", len(embeddings))
print("Embedding size:", len(embeddings[0]))

RAG data now looks conceptually like this
 Chunk 0  
            
 benefits_policy.pdf
          
 page 1                     
"Employee Benefits Policy..."

                             
 Embedding:                   
 [0.012, -0.034, ..., 0.123]

In [ ]:
#Ask a question Let's pretend a user asks your RAG system:"How long do employees have to work before they are eligible for benefits?"
question = "How the Respectful Workplace Employees must treat colleagues?"

print("Question embedding size:", len(question_embedding))
print(question_embedding[:10]) #The question has now been converted into the same type of vector as your PDF chunks.

step6-384-number vectors
                     ↓
        

In [ ]:
#Compare the question with your chunks
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(
    [question_embedding],
    embeddings
)[0]

print(similarities) #You'll get 11 similarity scores, one for each chunk.Higher similarity More semantically related,More likely to contain the answer

In [ ]:
#Find the most relevant chunk
best_index = similarities.argmax()

print("Best matching chunk:", best_index)
print("Similarity score:", similarities[best_index])
print()
print("Source:", chunks[best_index]["source"])
print("Page:", chunks[best_index]["page"])
print()
print("Text:")
print(chunks[best_index]["text"])

In [ ]:
top_k = 3

top_indices = similarities.argsort()[-top_k:][::-1]

for rank, index in enumerate(top_indices, start=1):

    print("=" * 60)
    print("Rank:", rank)
    print("Similarity:", similarities[index])
    print("Source:", chunks[index]["source"])
    print("Page:", chunks[index]["page"])
    print()
    print(chunks[index]["text"])

In [ ]:
context = "\n\n".join(
    chunks[index]["text"]
    for index in top_indices
)

print(context)

step 10

In [ ]:
#10.1 Install the library
!pip install -q -U google-genai

In [ ]:
#10.2 Import OpenAI
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

In [ ]:
#10.3 Create your API client
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini client is ready!")

In [ ]:
for model in client.models.list():
    print(model.name)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain what RAG is in one simple sentence."
)

print(response.text)

In [ ]:
print(context)

In [ ]:
prompt = f"""
You are a company policy assistant.

Answer the user's question using only the policy context provided below.

If the answer cannot be found in the policy context, say:
"I don't have enough information in the provided policies."

Policy context:
{context}

User question:
{question}
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print("Answer:")

print(response.text)

In [ ]:
import time

def ask_gemini(prompt, max_retries=3):

    for attempt in range(max_retries):

        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt
            )

            return response.text

        except Exception as e:

            if "503" in str(e) or "UNAVAILABLE" in str(e):

                wait_time = 5 * (2 ** attempt)

                print(
                    f"Gemini is temporarily busy. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                raise e

    return "Gemini is temporarily unavailable. Please try again later."

In [ ]:
def ask_question(question, top_k=3):

    # 1. Convert the question into an embedding
    question_embedding = model.encode(question)

    # 2. Calculate similarity between question and all chunks
    similarities = cosine_similarity(
        [question_embedding],
        embeddings
    )[0]

    # 3. Find the top matching chunks
    top_indices = similarities.argsort()[-top_k:][::-1]

    # 4. Create the context
    context = "\n\n".join(
        chunks[index]["text"]
        for index in top_indices
    )

    # 5. Create the prompt for Gemini
    prompt = f"""
You are a company policy assistant.

Answer the user's question using ONLY the policy context below.

If the answer cannot be found in the context, say:
"I don't have enough information in the provided policies."

Policy context:
{context}

User question:
{question}
"""

    # 6. Ask Gemini
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    # 7. Return the answer
    return response.text

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

In [ ]:
answer = ask_question(
    "How long do employees have to work before they are eligible for benefits?"
)

print(answer)